# Bitcoin Price Predictor — Live Snowflake Demo

This notebook runs **entirely offline** inside Snowflake: it reads the BTC/USDT
1-second kline data already committed to this repo under `data/raw_parquet/`
(cloned in via Snowflake's native Git integration), optionally (re)trains the
model using all 4 GPUs on this compute pool, and renders the same **1 fps
live-updating chart** described in the project plan:

- Actual market price (solid line)
- Model's predicted price, 25 minutes ahead (dotted line)
- Rolling accuracy bar, scored against the real stored market data (never the
  simulated feed) for that same timestamp

> No internet access is required to run this notebook. The only network
> activity anywhere in this project is the one-time `scripts/download_data.py`
> run (done already, outside Snowflake) that produced the Parquet files you
> see in `data/raw_parquet/`.


## 0. Environment check (GPUs, packages)

In [ ]:
import torch, os, sys
sys.path.insert(0, os.path.abspath(".."))  # repo root, if notebook lives in notebooks/
sys.path.insert(0, os.path.abspath("."))

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"({torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB)")


## 1. (Optional) Train from scratch on all 4 GPUs

This shells out to `train_distributed.sh`, which launches
`src/online_trainer.py` under `torchrun --nproc_per_node=4` — true
multi-GPU DistributedDataParallel training, continuously streaming through
the historical data second-by-second (no fixed epochs), with checkpoints
written to `checkpoints/`.

**Skip this cell** if you already have a checkpoint (e.g. `checkpoints/ema_final.pt`)
and just want to see the live visualization in section 2.


In [ ]:
# Uncomment to launch full distributed training (can run for a long time --
# it streams through the ENTIRE downloaded history once; pass --loop to
# src/online_trainer.py via EXTRA_ARGS for a never-ending "live" job).
#
# %cd ..
# !NUM_GPUS=4 BATCH_SIZE=48 CHECKPOINT_DIR=checkpoints/run_live bash train_distributed.sh
# %cd notebooks


## 2. Load data + model

In [ ]:
import glob
import numpy as np
import torch

from src.dataset import available_months, load_full_history, load_months
from src.online_trainer import precompute_shard_tensors, build_training_windows
from src.price_predictor import PricePredictor, PredictorConfig
from src.visualize import LiveChart, accuracy_from_error

months = available_months()
print(f"{len(months)} months available: {months[0]} .. {months[-1]}")

# Full history fits comfortably in Snowflake's 100GB RAM. For a quick test,
# set a smaller slice via months[:k].
md = load_full_history()
print(f"Loaded {len(md):,} seconds ({md.datetimes[0]} .. {md.datetimes[len(md)-1]})")


In [ ]:
CONTEXT = 12 * 60 * 60   # 12 hours, per spec
HORIZON = 25 * 60        # 25 minutes, per spec

cfg = PredictorConfig()  # the full "large" architecture
cfg.context, cfg.horizon = CONTEXT, HORIZON
model = PricePredictor(cfg)

ckpts = sorted(glob.glob("../checkpoints/**/*.pt", recursive=True))
if ckpts:
    ckpt_path = ckpts[-1]
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    print(f"Loaded checkpoint: {ckpt_path} (step {ckpt.get('step')})")
else:
    print("No checkpoint found yet -> using randomly-initialized weights "
          "(run section 1 first for a real trained model).")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print("Running inference on:", device)


## 3. Pick a random day (2020-2026) and precompute its features

(Feature extraction -- the "Market Analyser" -- runs on CPU, as specified.)


In [ ]:
import random

SEED = random.randint(0, 10_000)
n = len(md)
lo, hi = CONTEXT, n - HORIZON
day_start = random.randint(lo, hi - 86400) if hi - 86400 > lo else random.randint(lo, hi)
print(f"Replaying from {md.datetimes[day_start]} (seed={SEED})")

# Precompute a local shard covering [day_start - CONTEXT, day_start + 86400 + HORIZON]
shard_lo = max(0, day_start - CONTEXT)
shard_hi = min(n, day_start + 86400 + HORIZON)
shard = precompute_shard_tensors(md, shard_lo, shard_hi)
local_start = day_start - shard_lo
demo_len = min(86400, shard_hi - shard_lo - CONTEXT - HORIZON)
print(f"Will animate {demo_len} simulated seconds (~{demo_len/60:.0f} minutes of market time)")


## 4. Live 1 fps animated replay

Updates the chart in-place once per simulated second: actual price (solid),
predicted +25min price (dotted), and a rolling accuracy bar compared against
the REAL stored market data (not the simulated feed) for that timestamp.


In [ ]:
import time
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

chart = LiveChart(title=f"BTC/USDT Live Replay starting {md.datetimes[day_start]}")
pending_predictions = {}

FPS_SECONDS = 1.0           # 1 fps, per spec
MAX_FRAMES = min(demo_len, 1800)  # cap a single cell run to ~30 simulated minutes;
                                   # re-run the cell (it keeps going) for more

with torch.no_grad():
    for step in range(MAX_FRAMES):
        idx = local_start + step
        raw, feat, _ = build_training_windows(shard, idx, CONTEXT, HORIZON)
        raw_t = torch.from_numpy(raw).unsqueeze(0).to(device)
        feat_t = torch.from_numpy(feat).unsqueeze(0).to(device)

        t0 = time.time()
        cum_log_ret = model(raw_t, feat_t).squeeze(0).cpu().numpy()
        last_price = float(shard["close"][idx])
        pred_path = last_price * np.exp(cum_log_ret)

        abs_idx = shard_lo + idx
        for h, p in enumerate(pred_path, start=1):
            pending_predictions[idx + h] = (float(p), abs_idx + h)

        accuracy_now = None
        if idx in pending_predictions:
            made_pred, _ = pending_predictions.pop(idx)
            accuracy_now = accuracy_from_error(made_pred, float(shard["close"][idx]))

        chart.push_frame(shard_lo + idx, float(shard["close"][idx]), pred_path, accuracy_now)

        clear_output(wait=True)
        fig = chart.render()
        display(fig)

        elapsed = time.time() - t0
        if elapsed < FPS_SECONDS:
            time.sleep(FPS_SECONDS - elapsed)

plt.close(chart.fig)
print("Done. Re-run this cell to continue from where it left off (local_start keeps advancing) "
      "or re-run section 3 for a fresh random day.")


## Notes

- **Offline-safe**: every cell above only touches local files already in the
  repo (`data/raw_parquet/*.parquet`, `checkpoints/*.pt`). Nothing here makes
  a network call.
- **Stability**: the Price Predictor bounds its own per-second output change
  (tanh-capped log-return deltas + a smoothness loss term during training),
  which is why the dotted prediction line doesn't jump erratically even
  though it's recomputed from scratch every second.
- **Scaling up**: this notebook uses whatever checkpoint exists under
  `checkpoints/`. To train the full-size model from scratch across all 4 A10
  GPUs, run section 1 (or `bash train_distributed.sh` from a terminal) and
  come back here once checkpoints start appearing.
